In [5]:

from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT = Path("/kaggle/working/ubiris_v2_project")
PRE_DIR = PROJECT / "notebook02_preprocessing"
IMG_DIR = PRE_DIR / "images_224"
META_PATH = PRE_DIR / "UBIRIS_V2_preprocessed_metadata.csv"
QUALITY_PATH = PROJECT / "notebook03_quality_assessment/UBIRIS_V2_image_quality_metrics.csv"
OUT_DIR = PROJECT / "notebook04_baseline_recognition"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert IMG_DIR.exists(), f"Missing images: {IMG_DIR}"
assert META_PATH.is_file(), f"Missing metadata: {META_PATH}"

df = pd.read_csv(META_PATH)
required_cols = {"filename", "subject", "processed_path"}
assert required_cols.issubset(df.columns), f"Metadata columns missing: {required_cols - set(df.columns)}"

df["image_path"] = df["filename"].map(lambda x: str(IMG_DIR / (Path(str(x)).stem + ".png")))
df = df[df["image_path"].map(lambda p: Path(p).is_file())].copy()
df["subject"] = df["subject"].astype(str)
df = df[df["subject"].str.lower().ne("nan")].copy()

# Encode filename-parsed subjects consistently; these are provisional labels.
classes = sorted(df["subject"].unique())
class_to_idx = {name: i for i, name in enumerate(classes)}
df["label"] = df["subject"].map(class_to_idx).astype(int)

# Split by image while keeping each class represented where possible.
train_parts, val_parts, test_parts = [], [], []
for subject, group in df.groupby("subject"):
    group = group.sample(frac=1, random_state=SEED)
    n = len(group)
    if n >= 3:
        n_test = max(1, round(n * 0.15))
        n_val = max(1, round(n * 0.15))
        if n_test + n_val >= n:
            n_test, n_val = 1, 1
        test_parts.append(group.iloc[:n_test])
        val_parts.append(group.iloc[n_test:n_test+n_val])
        train_parts.append(group.iloc[n_test+n_val:])
    else:
        # Too few images to create all three partitions reliably.
        train_parts.append(group)

train_df = pd.concat(train_parts).sample(frac=1, random_state=SEED)
val_df = pd.concat(val_parts) if val_parts else pd.DataFrame(columns=df.columns)
test_df = pd.concat(test_parts) if test_parts else pd.DataFrame(columns=df.columns)

for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    part.to_csv(OUT_DIR / f"UBIRIS_V2_{name}_split.csv", index=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 58)
print("UBIRIS V2 — NOTEBOOK 04 SETUP")
print("=" * 58)
print("PyTorch:", torch.__version__)
print("Device:", device)
print("CUDA GPUs detected:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
print("Images with existing preprocessed files:", len(df))
print("Provisional filename-parsed subject classes:", len(classes))
print("Train / validation / test:", len(train_df), "/", len(val_df), "/", len(test_df))
print("Classes with at least 3 images:", int((df.groupby("subject").size() >= 3).sum()))
print("Split CSVs saved in:", OUT_DIR)
print("\nImportant: labels are parsed from filenames and are not independently identity-verified.")


UBIRIS V2 — NOTEBOOK 04 SETUP
PyTorch: 2.11.0+cu128
Device: cuda
CUDA GPUs detected: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Images with existing preprocessed files: 5799
Provisional filename-parsed subject classes: 260
Train / validation / test: 4243 / 778 / 778
Classes with at least 3 images: 260
Split CSVs saved in: /kaggle/working/ubiris_v2_project/notebook04_baseline_recognition

Important: labels are parsed from filenames and are not independently identity-verified.


In [6]:

from pathlib import Path
import json, time, random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ---------- Reproducibility and device ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

PROJECT = Path("/kaggle/working/ubiris_v2_project")
OUT_DIR = PROJECT / "notebook04_baseline_recognition"
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(OUT_DIR / "UBIRIS_V2_train_split.csv")
val_df = pd.read_csv(OUT_DIR / "UBIRIS_V2_validation_split.csv")
test_df = pd.read_csv(OUT_DIR / "UBIRIS_V2_test_split.csv")

# Consistent class mapping across all splits
all_labels = sorted(set(train_df["label"]) | set(val_df["label"]) | set(test_df["label"]))
num_classes = max(all_labels) + 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpu = torch.cuda.device_count() if device.type == "cuda" else 0
print("Device:", device, "| GPUs detected:", n_gpu)
if n_gpu:
    for i in range(n_gpu):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# ---------- Dataset ----------
class UBIRISDataset(Dataset):
    def __init__(self, frame, train=False):
        self.frame = frame.reset_index(drop=True)
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomRotation(5) if train else transforms.Lambda(lambda x: x),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row["image_path"]) as im:
            im = im.convert("L")
            x = self.transform(im)
        return x, int(row["label"])

train_ds = UBIRISDataset(train_df, train=True)
val_ds = UBIRISDataset(val_df)
test_ds = UBIRISDataset(test_df)

BATCH_SIZE = 64
NUM_WORKERS = 2
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

# ---------- Model ----------
model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

if n_gpu >= 2:
    model = nn.DataParallel(model)
    print("Using DataParallel across both T4 GPUs.")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

EPOCHS = 8
PATIENCE = 3
best_val_acc = -1.0
history = []
best_path = OUT_DIR / "UBIRIS_V2_baseline_resnet18_best.pth"
start = time.time()
stale_epochs = 0

def evaluate(model, loader):
    model.eval()
    losses, y_true, y_pred = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                logits = model(x)
                loss = criterion(logits, y)
            losses.append(loss.item())
            y_true.extend(y.cpu().numpy())
            y_pred.extend(logits.argmax(1).cpu().numpy())
    return float(np.mean(losses)), accuracy_score(y_true, y_pred), y_true, y_pred

for epoch in range(1, EPOCHS + 1):
    model.train()
    batch_losses = []
    t0 = time.time()
    for step, (x, y) in enumerate(train_loader, 1):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        batch_losses.append(loss.item())

        if step % 25 == 0 or step == len(train_loader):
            print(f"Epoch {epoch}/{EPOCHS} | batch {step}/{len(train_loader)} | loss {loss.item():.4f}")

    val_loss, val_acc, _, _ = evaluate(model, val_loader)
    train_loss = float(np.mean(batch_losses))
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "validation_loss": val_loss,
        "validation_accuracy": val_acc,
        "epoch_seconds": round(time.time() - t0, 2)
    })
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        stale_epochs = 0
        base_model = model.module if isinstance(model, nn.DataParallel) else model
        torch.save({
            "model_state_dict": base_model.state_dict(),
            "num_classes": num_classes,
            "class_labels": all_labels,
            "input_size": [224, 224],
            "input_mode": "grayscale",
            "seed": SEED
        }, best_path)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print("Early stopping.")
            break

# ---------- Final test evaluation using best checkpoint ----------
checkpoint = torch.load(best_path, map_location=device, weights_only=False)
base_model = model.module if isinstance(model, nn.DataParallel) else model
base_model.load_state_dict(checkpoint["model_state_dict"])
base_model.eval()

test_loss, test_acc, y_true, y_pred = evaluate(base_model, test_loader)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)

metrics = {
    "dataset": "UBIRIS V2",
    "device": str(device),
    "gpu_count_detected": n_gpu,
    "training_images": len(train_ds),
    "validation_images": len(val_ds),
    "test_images": len(test_ds),
    "num_classes": num_classes,
    "epochs_completed": len(history),
    "best_validation_accuracy": best_val_acc,
    "test_loss": test_loss,
    "test_accuracy": float(test_acc),
    "test_macro_precision": float(precision),
    "test_macro_recall": float(recall),
    "test_macro_f1": float(f1),
    "identity_labels_independently_verified": False,
    "note": "Exploratory closed-set image-level split; filename-parsed labels are provisional."
}

pd.DataFrame(history).to_csv(OUT_DIR / "UBIRIS_V2_baseline_training_history.csv", index=False)
pd.DataFrame([metrics]).to_csv(OUT_DIR / "UBIRIS_V2_baseline_test_metrics.csv", index=False)
(OUT_DIR / "UBIRIS_V2_baseline_test_metrics.json").write_text(json.dumps(metrics, indent=2))

print("\nBASELINE RESULTS")
print(json.dumps(metrics, indent=2))
print("\nSaved checkpoint:", best_path)
print("Elapsed minutes:", round((time.time() - start) / 60, 2))


Device: cuda | GPUs detected: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Using DataParallel across both T4 GPUs.
Epoch 1/8 | batch 25/67 | loss 5.7289
Epoch 1/8 | batch 50/67 | loss 5.4200
Epoch 1/8 | batch 67/67 | loss 5.1492
Epoch 1: train_loss=5.4611, val_loss=5.1181, val_acc=0.0270
Epoch 2/8 | batch 25/67 | loss 5.0186
Epoch 2/8 | batch 50/67 | loss 4.9922
Epoch 2/8 | batch 67/67 | loss 4.4983
Epoch 2: train_loss=4.8711, val_loss=4.6692, val_acc=0.0630
Epoch 3/8 | batch 25/67 | loss 4.2571
Epoch 3/8 | batch 50/67 | loss 4.0591
Epoch 3/8 | batch 67/67 | loss 3.7699
Epoch 3: train_loss=4.2030, val_loss=4.6519, val_acc=0.0578
Epoch 4/8 | batch 25/67 | loss 3.4116
Epoch 4/8 | batch 50/67 | loss 3.2971
Epoch 4/8 | batch 67/67 | loss 3.9088
Epoch 4: train_loss=3.4618, val_loss=4.6464, val_acc=0.1144
Epoch 5/8 | batch 25/67 | loss 2.6078
Epoch 5/8 | batch 50/67 | loss 2.2011
Epoch 5/8 | batch 67/67 | loss 2.9933
Epoch 5: train_loss=2.6719, val_loss=3.1325, val_acc=0.2494
Epoch 6/8 | batch 25/67 | 


# Final Conclusion — UBIRIS V2 Baseline Recognition

## Experiment Overview
A grayscale ResNet18 baseline classifier was trained on the UBIRIS V2 dataset using two Tesla T4 GPUs with PyTorch DataParallel. The model used 224 × 224 grayscale images and was trained for 8 epochs.

## Results
- Training images: 4,243
- Validation images: 778
- Test images: 778
- Provisional filename-parsed classes: 260
- Best validation accuracy: 47.69%
- Test accuracy: 47.43%
- Macro precision: 42.96%
- Macro recall: 44.21%
- Macro F1-score: 38.97%
- Training time: approximately 1.28 minutes

The best validation accuracy was achieved at epoch 7. The best checkpoint and training history were saved for later experiments.

## Limitations
1. Subject labels were parsed from filenames and have not been independently verified.
2. The experiment is exploratory and does not establish verified identity recognition.
3. The image-level split may allow images from the same provisional class in different splits; therefore, this is not an unseen-subject generalization evaluation.
4. Results should be interpreted within this dataset and experimental setup.

## Conclusion
This baseline establishes a reference point for subsequent UBIRIS V2 augmentation and recognition experiments. Future comparisons should preserve the same evaluation split and report label limitations explicitly.


In [7]:

from pathlib import Path
import zipfile
from IPython.display import FileLink, display

BASE = Path("/kaggle/working/ubiris_v2_project")
RESULTS = BASE / "notebook04_baseline_recognition"
ZIP_PATH = Path("/kaggle/working/UBIRIS_V2_Notebook04_Results.zip")

if not RESULTS.exists():
    raise FileNotFoundError(f"Results directory not found: {RESULTS}")

files = [p for p in RESULTS.rglob("*") if p.is_file() and p != ZIP_PATH]

if not files:
    raise RuntimeError("No Notebook 04 output files found.")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in files:
        zf.write(path, path.relative_to(RESULTS))

print(f"ZIP created: {ZIP_PATH.name}")
print(f"Files included: {len(files)}")
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")
display(FileLink(str(ZIP_PATH)))


ZIP created: UBIRIS_V2_Notebook04_Results.zip
Files included: 7
ZIP size: 40.20 MB


/kaggle/working/UBIRIS_V2_Notebook04_Results.zip